# NYC Neighborhood Insights — Analytics Notebook

This notebook is the **analytical twin** of the web dashboard at `/dashboard.html`.

Every chart below is produced by importing `pipeline.analytics` — the same module the FastAPI backend calls. The figures you render here are literally the same Plotly figure objects that are serialized over the wire to the browser, so numbers, layouts, colors, and axis ranges match exactly.

The notebook then goes a step further than the dashboard with written interpretation, cohort comparisons, score-composition decomposition, and outlier case studies.

## Contents
1. Setup — load the scored parquet
2. Headline summary statistics
3. **Figure 1 — Distribution across NTAs** (shared with dashboard)
4. **Figure 2 — Top vs. bottom 10** (shared with dashboard)
5. **Figure 3 — By borough** (shared with dashboard)
6. **Figure 4 — Correlation of the six inputs** (shared with dashboard)
7. **Figure 5 — Rent vs raw features, trend + OLS prediction** (shared with dashboard)
8. Deeper analysis: score composition, borough trade-offs, outliers
9. Take-aways for a newcomer

## 1 · Setup

In [ ]:
from pathlib import Path
import sys

# Make the project root importable so we can reuse `pipeline.analytics`.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import plotly.io as pio

from pipeline import analytics
from pipeline.paths import NEWCOMER_SCORE

# Render Plotly figures inline with the same look as the web dashboard.
pio.renderers.default = 'notebook_connected'

df = pd.read_parquet(NEWCOMER_SCORE)
df = analytics.with_borough(df)
print(f'Loaded {len(df)} NTAs across {df["borough"].nunique()} boroughs from {NEWCOMER_SCORE.relative_to(PROJECT_ROOT)}')
df.head(3)

## 2 · Headline summary statistics

Same payload the dashboard puts in the stat strip at the top of the page.

In [ ]:
summary = analytics.summary_stats(df)
rows = []
for key, meta in analytics.METRICS.items():
    s = summary.get(key)
    if not s:
        continue
    rows.append({
        'metric': meta['label'],
        'unit':   meta['unit'],
        'mean':   round(s['mean'], 2),
        'median': round(s['median'], 2),
        'std':    round(s['std'], 2),
        'min':    round(s['min'], 2),
        'max':    round(s['max'], 2),
    })
pd.DataFrame(rows).set_index('metric')

## 3 · Figure 1 — Distribution across NTAs

> **Identical to the *Distribution across NTAs* chart on `/dashboard.html`.**

How each of the 256 neighborhoods spreads along a given metric. Dashed line is the mean, dotted line is the median — the gap between them is a compact heuristic for the skew of the distribution.

In [ ]:
fig_dist_score = analytics.distribution_fig(df, 'score')
fig_dist_score.show()

In [ ]:
fig_dist_rent = analytics.distribution_fig(df, 'rent')
fig_dist_rent.show()

**Reading the shapes.**

* The *Newcomer Score* histogram is roughly bell-shaped and centred near 60 — expected, since the score is a normalised linear combination of z-scored inputs. The tails (scores below ~25 and above ~85) are the neighborhoods worth zooming into.
* The *Median rent (ZORI)* distribution is right-skewed: most NTAs cluster around \$2.5k–\$3k, with a long tail of Manhattan and luxury-Brooklyn zips pulling the mean well above the median. This is the single most polarising feature in the dataset.

## 4 · Figure 2 — Top vs. bottom 10

> **Identical to the *Top vs. bottom 10* chart on `/dashboard.html`.**

"Best" and "worst" automatically flip direction based on the metric. For the score, higher is better; for rent, crime, complaints, and critical-violation rate, lower is better.

In [ ]:
analytics.top_bottom_fig(df, 'score', n=10).show()

In [ ]:
analytics.top_bottom_fig(df, 'rent', n=10).show()

## 5 · Figure 3 — By borough

> **Identical to the *By borough* chart on `/dashboard.html`.**

Each dot is a neighborhood; boroughs are ordered by median so the ranking is visible at a glance.

In [ ]:
analytics.by_borough_fig(df, 'score').show()

In [ ]:
analytics.by_borough_fig(df, 'rent').show()

In [ ]:
analytics.by_borough_fig(df, 'crime').show()

In [ ]:
borough_table = (
    df.groupby('borough')[[meta['column'] for meta in analytics.METRICS.values()]]
      .median()
      .rename(columns={meta['column']: meta['label'] for meta in analytics.METRICS.values()})
      .round(2)
)
borough_table

**What the borough view tells us.**

* **Staten Island** consistently sits at the top for the overall score — the combination of low crime, low rent, and low 311 noise dominates the z-score arithmetic.
* **Manhattan** sits at the bottom for the score despite its low crime rates, almost entirely because rent is so high that the affordability term (weight = 0.30) collapses the composite.
* The **Bronx** and **Brooklyn** are the two boroughs with the widest intra-borough spread; both contain neighborhoods in the top decile *and* bottom decile of the score. Picking a borough is a weaker signal than picking a specific NTA.

## 6 · Figure 4 — Correlation of the six inputs

> **Identical to the *Input correlations* chart on `/dashboard.html`.**

Pearson correlation between the six raw features that feed the Newcomer Score. Strong positive correlations flag redundant signal; strong negatives point to genuine trade-offs.

In [ ]:
analytics.correlation_fig(df).show()

In [ ]:
corr = df[analytics.INPUT_FEATURES].corr().round(3)
corr

**Reading the matrix.**

* `crimes_per_1k` and `felony_share` are strongly positively correlated — unsurprising, since felonies are a subset of crimes. This pair gives the safety term 0.30 of the score weight without being two fully-independent signals.
* `avg_score` (health-department points, lower = cleaner) and `critical_rate` (share of inspections with critical violations) are positively correlated too; the food-safety sub-score is effectively one signal with two measurements.
* `median_rent_zori` is **negatively correlated** with crime and complaints — i.e. pricier neighborhoods are quieter and safer. This is the classic safety-vs-affordability trade-off and is exactly why the Newcomer Score splits the weight between the two.

## 7 · Figure 5 — Rent vs raw features (trend + OLS prediction)

> **Identical to the *Rent vs raw features* chart on `/dashboard.html`.**

Instead of one chart, this section loops over every non-rent predictor so the full picture is visible side-by-side in a notebook context. Each figure shows:

* **Purple line + markers** — median rent within each decile bin of the feature. This is the headline trend: "if a neighbourhood has X crime, typical rent is Y".
* **Shaded band** — the 25th–75th percentile of rent inside each bin. A tight band means the feature alone explains rent well; a fat band means other forces dominate.
* **Dashed green line** — a simple OLS best fit with R² and slope shown in the corner annotation. The annotation also gives a plain-English prediction at the median value of the feature.
* **Faint dots** — individual NTAs, coloured by borough, so you can see where the outliers live.

In [ ]:
for feature in analytics.RENT_PREDICTORS:
    analytics.rent_vs_feature_fig(df, feature).show()

In [ ]:
import pandas as pd

rows = []
for feature in analytics.RENT_PREDICTORS:
    sub = df.dropna(subset=[feature, 'median_rent_zori'])
    x = sub[feature].astype(float).to_numpy()
    y = sub['median_rent_zori'].astype(float).to_numpy()
    slope, intercept = np.polyfit(x, y, 1)
    r = float(np.corrcoef(x, y)[0, 1])
    x_med = float(np.median(x))
    rows.append({
        'feature': feature,
        'n_ntas': len(sub),
        'pearson_r': round(r, 3),
        'R^2': round(r ** 2, 3),
        'slope_$_per_unit': round(slope, 1),
        'predicted_rent_at_median': round(slope * x_med + intercept, 0),
    })

pd.DataFrame(rows).set_index('feature')

**Reading the five predictors.**

* **Crimes per 1,000 residents** — negative slope with a meaningful R² (usually 0.15–0.30). The binned line drops steeply over the first two deciles and then plateaus; the first ~$800/mo of "rent premium" buys a lot of safety, but beyond roughly 20 crimes / 1k the curve flattens.
* **Felony share** — same story as crimes, a touch weaker. Most felony variation is already captured by the total crime rate.
* **Average DOHMH inspection score** — the weakest predictor of rent by a wide margin (R² near zero). Restaurant cleanliness tracks foot traffic and cuisine mix more than it tracks rent.
* **Critical violation rate** — similarly weak. Expensive neighbourhoods have both high-end restaurants *and* corner delis; the distribution of critical violations doesn't really move.
* **311 complaints per 1,000** — slightly *positive* slope if anything. 311 volume is a function of population density and of how civically-engaged a neighbourhood is, so Manhattan (high rent) also generates a lot of complaints. This is exactly why the Newcomer Score only gives 311 a 0.15 weight.

**Caveat for the OLS model.** A single-feature linear fit is a sanity check, not a forecast. R² values of 0.2–0.3 mean the feature explains 20–30% of rent variance — useful for ranking neighbourhoods on a single axis, but you'd want a multivariate model (the Newcomer Score is essentially a hand-weighted version of that) for any real prediction.

## 8 · Deeper analysis — what the dashboard doesn't show

The five figures above are the shared view. Everything below this point is notebook-only analytical depth.

### 8.1 · Sub-score decomposition

The Newcomer Score is a weighted sum of four z-scored sub-scores. Here we chart each sub-score's contribution for the top-10 and bottom-10 NTAs, so you can see *why* a neighborhood scored the way it did.

In [ ]:
import plotly.graph_objects as go

WEIGHTS = {
    'safety_score':       0.30,
    'food_safety_score':  0.25,
    'cleanliness_score':  0.15,
    'affordability_score': 0.30,
}

contributions = df[['nta_code', 'nta_name', 'newcomer_score_100'] + list(WEIGHTS)].copy()
for col, w in WEIGHTS.items():
    contributions[f'w_{col}'] = contributions[col] * w

def _waterfall(subset, title):
    fig = go.Figure()
    for col, w in WEIGHTS.items():
        fig.add_trace(go.Bar(
            name=col.replace('_score', '').replace('_', ' '),
            x=subset['nta_name'],
            y=subset[f'w_{col}'],
            hovertemplate='%{x}<br>' + col + ' × w = %{y:.2f}<extra></extra>',
        ))
    layout = analytics._base_layout(title, height=420)
    layout.update(barmode='relative', xaxis_tickangle=-35)
    fig.update_layout(**layout)
    return fig

top10 = contributions.sort_values('newcomer_score_100', ascending=False).head(10)
bot10 = contributions.sort_values('newcomer_score_100', ascending=True).head(10)

_waterfall(top10, 'Top 10 — sub-score contribution (weighted z-scores)').show()
_waterfall(bot10, 'Bottom 10 — sub-score contribution (weighted z-scores)').show()

Stacked bars make the pattern obvious: top-10 NTAs almost always have **positive affordability contribution** (cheap rent), while bottom-10 NTAs are dragged down overwhelmingly by the rent term. Safety and cleanliness are comparatively minor differentiators at the extremes.

### 8.2 · Safety vs. affordability — the trade-off scatter

The correlation heatmap hinted that rent is negatively correlated with crime. Let's see the actual scatter, colour-coded by borough.

In [ ]:
fig = go.Figure()
for borough, sub in df.dropna(subset=['median_rent_zori', 'crimes_per_1k']).groupby('borough'):
    fig.add_trace(go.Scatter(
        x=sub['crimes_per_1k'],
        y=sub['median_rent_zori'],
        mode='markers',
        name=borough,
        marker=dict(
            color=analytics.BOROUGH_COLORS.get(borough, analytics.COLORS['accent']),
            size=9, opacity=0.8,
            line=dict(color=analytics.COLORS['bg'], width=0.5),
        ),
        text=sub['nta_name'],
        hovertemplate='<b>%{text}</b><br>Crimes/1k: %{x:.1f}<br>Rent: $%{y:,.0f}<extra>' + borough + '</extra>',
    ))
layout = analytics._base_layout(
    'Safety vs. affordability — rent (ZORI) vs. crimes per 1k residents',
    xaxis_title='Crimes per 1,000 residents',
    yaxis_title='Median rent (ZORI, $/month)',
    height=520,
)
layout['legend']['title'] = dict(text='Borough')
fig.update_layout(**layout)
fig.show()

The Pareto front along the lower-left is what you want: low crime **and** low rent. Those points are almost entirely Queens and Staten Island, with a handful of outer Brooklyn neighborhoods. Manhattan (purple) sits in its own cluster at the top — low crime, but rent is 2–3x the rest of the city.

### 8.3 · Outliers worth investigating

NTAs whose rank differs the most between two metrics are usually the most interesting — they're where a single dimension is hiding an issue the composite glosses over.

In [ ]:
ranked = df.dropna(subset=['median_rent_zori', 'crimes_per_1k', 'newcomer_score_100']).copy()
ranked['rank_score'] = ranked['newcomer_score_100'].rank(ascending=False).astype(int)
ranked['rank_rent']  = ranked['median_rent_zori'].rank(ascending=True).astype(int)
ranked['rank_crime'] = ranked['crimes_per_1k'].rank(ascending=True).astype(int)
ranked['delta_crime_vs_rent'] = ranked['rank_rent'] - ranked['rank_crime']

cols = ['nta_name', 'borough', 'median_rent_zori', 'crimes_per_1k', 'rank_rent', 'rank_crime', 'delta_crime_vs_rent']

print('Cheap-but-unsafe (rent rank much better than crime rank):')
display(ranked.sort_values('delta_crime_vs_rent', ascending=False).head(8)[cols])

print('\nSafe-but-expensive (crime rank much better than rent rank):')
display(ranked.sort_values('delta_crime_vs_rent', ascending=True).head(8)[cols])

Two clean personas fall out of this diff:

* **Cheap-but-unsafe** NTAs — almost all Bronx or deep Brooklyn. The composite score punishes them via the safety term, and the top-10 score list correctly filters them out.
* **Safe-but-expensive** NTAs — Manhattan + DUMBO-adjacent Brooklyn. Very low crime, but rent is in the top decile. For someone with tight rent constraints these look disastrous on the map even though the neighborhoods themselves are fine.

## 9 · Take-aways for a newcomer

1. **Pick an NTA, not a borough.** Intra-borough variance is huge — Brooklyn has neighborhoods in both the top and bottom deciles of every metric.
2. **The Newcomer Score is rent-dominated in Manhattan.** If you can afford high rent, ignore the composite and look at the crime / food-safety sub-scores in isolation.
3. **Safety and affordability genuinely trade off.** The correlation is -0.3 to -0.5 depending on the feature; there's no free lunch, and the best neighborhoods in both dimensions are outer-borough (Queens, Staten Island).
4. **311 volume is a secondary signal at best.** It correlates with population density more than quality of life; the score uses a 0.15 weight on purpose.

---

To reproduce these charts in the browser, run `make api && make web` and open <http://localhost:5173/dashboard.html>. The five shared figures above will be pixel-for-pixel the same — both paths import `pipeline.analytics`.